# Creating Dynamic Agents

Utilizing wrapper middleware to change the model instance based on the situation the agent is facing. The middleware will adjust the model request when it's being executed.

I'm exploring two decorators that are the workhorses for custom middleware:
- dynamic_prompt --> a wrapper for creating custom middleware to change the system prompt for the agent based on context
- wrap_model_call --> wrapper for creating custom middleware to wrap the model call and change parameters like accessible tools and the model selected

In [18]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest, ModelResponse, wrap_model_call
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain_community.utilities import SQLDatabase

from typing import Dict, Any, Callable

from tavily import TavilyClient

from dataclasses import dataclass

from dotenv import load_dotenv

In [15]:
load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

Creating an agent with user language preferences that can be changed when invoked.

In [6]:
@dataclass
class LanguageContext:
    user_language: str

@dynamic_prompt
def user_language_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_language = request.runtime.context.user_language
    base_prompt = "You are a helpful assistant."

    if user_language != "English":
        return f"{base_prompt} only respond in {user_language}."
    elif user_language == "English":
        return base_prompt

In [7]:
agent = create_agent(
    model='claude-haiku-4-5',
    context_schema=LanguageContext,
    middleware=[user_language_prompt]
)

In [10]:
first_response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Irish")
)

print(first_response["messages"][-1].content)

Día duit! Táim go maith, go raibh maith agat as a fhiafraí. Conas atá tusa? 

(Hello! I'm well, thank you for asking. How are you?)


In [11]:
second_response = agent.invoke(
    {"messages": [HumanMessage(content="Hello, how are you?")]},
    context=LanguageContext(user_language="Spanish")
)

print(second_response["messages"][-1].content)

¡Hola! Estoy bien, gracias por preguntar. ¿Cómo estás tú? ¿En qué puedo ayudarte hoy?


## Role-Based Tool Access

Setting up tools for the agent to use

In [16]:
tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:
    """Query the database to get information"""
    try:
        return db.run(query)
    except Exception as e:
        return f"Error {e}"

Setting up a dataclass for the user's role

In [22]:
@dataclass
class UserRole:
    user_role: str = "external"

In [23]:
@wrap_model_call
def dynamic_tool_call(request: ModelRequest,
                      handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role

    if user_role == "internal":
        pass
    else:
        tools = [web_search]
        request = request.override(tools=tools)

    return handler(request)

In [24]:
tool_agent = create_agent(
    model='claude-haiku-4-5',
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [ ]:
dc_response1 = tool_agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database")]},
    context={"user_role": "internal"}
)

print(dc_response1["messages"][-1].content)

There are **275 artists** in the database.


In [26]:
dc_response2 = tool_agent.invoke(
    {"messages": HumanMessage(content="How many artists are in the database")},
    context={"user_role": "external"}
)

print(dc_response2["messages"][-1].content)

I don't have access to a database that I can query directly. To help you find this information, I would need more context:

1. **Which database are you referring to?** (e.g., Spotify, Last.fm, MusicBrainz, a local database, etc.)
2. **Do you have access to this database?** If so, you might need to use a database query tool or application to check the number of records.

If you're asking about a public music database or service, I can search for that information for you. Let me know which specific database you're interested in, and I'll be happy to help!
